# Route A — Object Detection with YOLO
## Revised core code samples, Sessions 1–9

**Mental model:** the camera **senses**, the program **thinks**, and the robot **acts**.

Teacher notes are marked with `TEACHER NOTE`. Remove those cells before giving a
student copy to the class.

Two bugs are still planted on purpose. They appear in Session 2 and Session 4 and
are identified in the teacher note immediately above them.


## Setup — teacher preparation

Before class:

- Keep `desk.jpg`, `messy_desk.jpg`, and `blank_wall.jpg` beside this notebook.
- Install `ultralytics`, `opencv-python`, and `ipython` in the selected VS Code environment.
- Run the setup once while online so `yolo26n.pt` downloads before the lesson.
- Give VS Code or Python permission to use the Mac camera.
- Before Sessions 8–9, connect the UGOT and create the `got` object using the school's tested SDK setup.
- Mark a clear robot test zone and rehearse the emergency-stop procedure.

> **TEACHER NOTE.** The model was trained before this lesson using large collections
> of labelled images. Students are using an already-trained model; their three class
> photographs are tests, not new training data.


### Teacher setup code — run this first


In [ ]:
from pathlib import Path
import time

import cv2
from IPython.display import Image, display
from ultralytics import YOLO

MODEL_IMAGE_SIZE = 320
MIN_CONFIDENCE = 0.5

model = YOLO("yolo26n.pt")
print("Model is ready.")


### A safe picture loader

This helper turns a missing or misspelled file into a clear message instead of
allowing the rest of the program to use an empty image.


In [ ]:
def load_picture(filename):
    picture_path = Path(filename)
    picture = cv2.imread(str(picture_path))

    if picture is None:
        raise FileNotFoundError(
            f"I could not open {filename}. Put it beside the notebook and check its name."
        )

    return picture


---
# Session 1 — Meet the AI that sees

**New idea:** one detection contains a class name.

**Predict:** if you had to teach a robot to find the water bottle in this picture,
what would you tell it?


### Run — find things in a picture


In [ ]:
picture = load_picture("desk.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    class_id = int(box.cls[0])
    name = model.names[class_id]
    print(name)
    


### Investigate — how many names does it know?

In Term 4 our fruit classifier knew **2** things, and we typed in **4** rows of
example data by hand. This model was trained before today on many labelled images.

How many class names does it know?


In [ ]:
print("Number of class names:", len(model.names))
print(model.names)


### Make

Photograph three objects around the room. Run the code on each photograph.
Record:

1. what the object really was;
2. what the model said;
3. whether the result was correct.


In [ ]:
picture = load_picture("my_photo.jpg")
results = model(picture, verbose=False)

if len(results[0].boxes) == 0:
    print("The model did not report an object.")
else:
    for box in results[0].boxes:
        print(model.names[int(box.cls[0])])


---
# Session 2 — How sure is it?

**New idea:** each detection also has a confidence score.

**Predict:** yesterday the model got some things wrong. Did its confidence score
give us a warning?


### Run — print the confidence too


In [ ]:
picture = load_picture("desk.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    confidence = float(box.conf[0])
    print(name, round(confidence, 2))


### Investigate

Confidence is a score between 0 and 1 that shows how strongly the model supports
its own detection. A larger score usually means the model is more sure, but it is
not a guarantee: a high-confidence answer can still be wrong.

Think of three neighbours voting as an analogy, not as the way YOLO literally works:

- all three agree — stronger support;
- two agree and one disagrees — weaker support.


> **TEACHER NOTE — PLANTED BUG (silent).**
> The threshold below is `0.9`. On many classroom objects, nothing prints.
> No error appears—the condition simply rejects every detection.
>
> After about four minutes, ask: *“The code ran, so what decision did it make?”*

### Modify — only show detections above the threshold


In [ ]:
picture = load_picture("desk.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    confidence = float(box.conf[0])

    if confidence > 0.9:
        print(name, round(confidence, 2))


### Make

Change the threshold and test several objects. Find:

- one detection above 0.8;
- one detection between 0.25 and 0.4.

Write one possible reason for the difference. Label it as a **hypothesis**, not a proven cause.


---
# Session 3 — Where is it?

**New idea:** a box can be described using two corners.

**Predict:** look at a box drawn on a picture. Give four numbers that describe
where it is.


### Run — print the corners


In [ ]:
picture = load_picture("desk.jpg")
results = model(picture, verbose=False)

if len(results[0].boxes) == 0:
    raise RuntimeError("No objects were detected. Try a clearer desk photograph.")

box = results[0].boxes[0]

x1 = int(box.xyxy[0][0])
y1 = int(box.xyxy[0][1])
x2 = int(box.xyxy[0][2])
y2 = int(box.xyxy[0][3])

print(x1, y1, x2, y2)


### Investigate

In Term 2, `cv2.boundingRect` gave us `x, y, w, h` — **one corner and a size**.

YOLO gives us `x1, y1, x2, y2` — **two corners**.

Same rectangle, different description.

**Your turn:** without looking it up, work out how to calculate `w` from `x1` and `x2`.


### Modify — width, height, and centre


In [ ]:
width = x2 - x1
height = y2 - y1
center_x = (x1 + x2) // 2

print("width:", width)
print("height:", height)
print("centre x:", center_x)


### Make — paper task, laptops closed

Use five printed frames. Read the coordinates by hand and calculate the width and
horizontal centre for each box.


---
# Session 4 — Many things at once

**New idea:** a result may contain zero, one, or many boxes.

**Predict:** the table has one bottle on it. Now I add three more objects. What
changes about what the code receives?


### Run — loop through everything it found


In [ ]:
picture = load_picture("messy_desk.jpg")
results = model(picture, verbose=False)

print("Things found:", len(results[0].boxes))

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    confidence = float(box.conf[0])
    print(name, round(confidence, 2))


### Investigate

Our earlier colour-tracking functions often selected exactly **one** contour.
YOLO is different: it may return zero boxes, one box, or many boxes.


> **TEACHER NOTE — PLANTED BUG (loud).**
> The cell below takes `boxes[0]` without checking. A truly blank result raises an
> `IndexError`. Use a prepared image that has already been tested to produce no boxes.
>
> Ask: *“Which bug was easier to find? Which one could be more dangerous on a robot?”*


In [ ]:
picture = load_picture("blank_wall.jpg")
results = model(picture, verbose=False)

box = results[0].boxes[0]
print(model.names[int(box.cls[0])])


### Modify — check before you look


In [ ]:
picture = load_picture("blank_wall.jpg")
results = model(picture, verbose=False)

if len(results[0].boxes) == 0:
    print("I cannot see a known object.")
else:
    box = results[0].boxes[0]
    print(model.names[int(box.cls[0])])


### Make — object census

Use `messy_desk.jpg`. Count how many of each detected class appears.

`.count()` is back from Term 4, Session 1.


In [ ]:
picture = load_picture("messy_desk.jpg")
results = model(picture, verbose=False)

names = []
for box in results[0].boxes:
    names.append(model.names[int(box.cls[0])])

print(names)
print("bottles:", names.count("bottle"))
print("cups:", names.count("cup"))


---
# Session 5 — Choosing one target

**New idea:** first filter acceptable boxes, then choose the strongest one.

**Predict:** if the picture contains three bottles, which one should the robot use?


### Run — keep only the boxes with the right name


In [ ]:
picture = load_picture("messy_desk.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    confidence = float(box.conf[0])

    if name == "bottle":
        print(name, round(confidence, 2))


### Investigate

Our program makes two separate decisions:

1. **Filter:** is this the correct name, and is its confidence high enough?
2. **Rank:** among acceptable boxes, which has the highest confidence?

Keeping those decisions separate makes the code easier to explain and tune.


### Make — `find_target`

This function is the deliverable for the first half of the term. Compare its name
and return value with `find_ball()` from Term 2.


In [ ]:
def find_target(frame, target_name, minimum_confidence=MIN_CONFIDENCE):
    results = model(frame, imgsz=MODEL_IMAGE_SIZE, verbose=False)

    best_box = None
    best_confidence = -1.0

    for box in results[0].boxes:
        name = model.names[int(box.cls[0])]
        confidence = float(box.conf[0])

        is_target = name == target_name
        is_confident_enough = confidence >= minimum_confidence
        is_best_so_far = confidence > best_confidence

        if is_target and is_confident_enough and is_best_so_far:
            best_box = box
            best_confidence = confidence

    if best_box is None:
        return None

    x1 = int(best_box.xyxy[0][0])
    y1 = int(best_box.xyxy[0][1])
    x2 = int(best_box.xyxy[0][2])
    y2 = int(best_box.xyxy[0][3])

    return x1, y1, x2 - x1, y2 - y1


### Check it works


In [ ]:
picture = load_picture("messy_desk.jpg")
target_box = find_target(picture, "bottle")

if target_box is None:
    print("No confident bottle detection.")
else:
    x, y, width, height = target_box
    print("Found a bottle at", x, y, "size", width, height)


---
# Session 6 — Bringing it to life

**New idea:** the same detector can receive a changing camera frame.

The camera is the **sense** part. `find_target` is the beginning of **think**.
The robot does not move yet.


### Make — live detection on the laptop camera


In [ ]:
camera = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)

try:
    if not camera.isOpened():
        raise RuntimeError("Camera did not open. Check macOS camera permission.")

    ok, frame = camera.read()
    if not ok or frame is None:
        raise RuntimeError("The camera opened but did not return a frame.")

    ok, frame_jpg = cv2.imencode(".jpg", frame)
    if not ok:
        raise RuntimeError("The first camera frame could not be displayed.")

    handle = display(Image(frame_jpg.tobytes()), display_id=True)

    while True:
        ok, frame = camera.read()
        if not ok or frame is None:
            print("Camera frame missed. Trying again...")
            time.sleep(0.1)
            continue

        target_box = find_target(frame, "bottle")

        if target_box is not None:
            x, y, width, height = target_box
            cv2.rectangle(frame, (x, y), (x + width, y + height), (0, 255, 0), 2)
            cv2.putText(
                frame,
                f"bottle width:{width}",
                (x, max(y - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2,
            )

        ok, frame_jpg = cv2.imencode(".jpg", frame)
        if ok:
            handle.update(Image(frame_jpg.tobytes()))

        time.sleep(0.05)

finally:
    camera.release()
    print("Camera closed.")


### Modify

Change `MODEL_IMAGE_SIZE` from `320` to `640`. Observe the speed and the detections,
then change it back. What did we gain, and what did we lose?

Stop the notebook cell when you have enough evidence.


---
# Session 7 — Box to control numbers

**New idea:** the box position becomes a signed steering error.

The robot still does not move today.

**Predict:** as a bottle moves from left to centre to right, what should its
steering error do?


In [ ]:
camera = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)

try:
    if not camera.isOpened():
        raise RuntimeError("Camera did not open. Check macOS camera permission.")

    while True:
        ok, frame = camera.read()
        if not ok or frame is None:
            time.sleep(0.1)
            continue

        target_box = find_target(frame, "bottle")

        if target_box is None:
            print("Show me a bottle.")
            time.sleep(0.5)
            continue

        x, y, width, height = target_box
        target_center_x = x + width // 2
        frame_center_x = frame.shape[1] // 2

        steering_error = target_center_x - frame_center_x
        size_error = 200 - width

        print(
            f"centre:{target_center_x}  "
            f"steering_error:{steering_error}  size_error:{size_error}"
        )
        time.sleep(0.2)

finally:
    camera.release()
    print("Camera closed.")


### Investigate — the reveal

Term 3 colour tracking:

```python
steering_error = target_center_x - frame_center_x
```

Today with YOLO:

```python
steering_error = target_center_x - frame_center_x
```

The controller did not learn a new language. We changed only where its box came from.

`size_error` is a **distance clue**, not true measured distance. A wider box usually
means the same object is closer, but different-sized objects cannot be compared directly.


---
# Session 8 — The robot approaches

**New idea:** bounded control turns the two errors into safe movement commands.

> **TEACHER SAFETY CHECK**
>
> 1. Test with wheels raised first.
> 2. Clear the floor and keep one student beside the emergency stop.
> 3. Begin with the low speed limit below.
> 4. The program stops after 20 seconds or after repeated camera failures.


In [ ]:
def target_centralizing_approaching_Kp(target_name, max_run_seconds=20):
    import numpy as np

    Kp_steering = 0.05
    Kp_approach = 0.25

    target_width = 200
    steering_dead_zone = 35
    size_dead_zone = 12
    speed_limit = 12
    maximum_frame_failures = 10

    def picture_bgr():
        picture_raw = got.read_camera_data()
        if picture_raw is None:
            return None
        return cv2.imdecode(
            np.frombuffer(picture_raw, np.uint8),
            cv2.IMREAD_COLOR,
        )

    def move_and_draw(frame, x, y, width, height):
        target_center_x = x + width // 2
        frame_center_x = frame.shape[1] // 2

        steering_error = target_center_x - frame_center_x
        size_error = target_width - width

        close_enough = abs(size_error) < size_dead_zone
        centered = abs(steering_error) < steering_dead_zone

        if centered and close_enough:
            got.mecanum_stop()
            return True

        steering_speed = int(Kp_steering * steering_error)
        approach_speed = int(Kp_approach * size_error)

        steering_speed = max(min(steering_speed, speed_limit), -speed_limit)
        approach_speed = max(min(approach_speed, speed_limit), -speed_limit)

        got.mecanum_move_xyz(steering_speed, approach_speed, 0)

        cv2.rectangle(frame, (x, y), (x + width, y + height), (0, 255, 0), 2)
        cv2.putText(
            frame,
            f"{target_name} steer:{steering_error} size:{size_error}",
            (x, max(y - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2,
        )
        return False

    handle = None
    frame_failures = 0
    started_at = time.monotonic()

    try:
        got.open_camera()

        while time.monotonic() - started_at < max_run_seconds:
            frame = picture_bgr()

            if frame is None:
                frame_failures += 1
                got.mecanum_stop()
                if frame_failures >= maximum_frame_failures:
                    print("Too many missing camera frames. Robot stopped.")
                    return False
                time.sleep(0.1)
                continue

            frame_failures = 0
            target_box = find_target(frame, target_name)

            if target_box is None:
                got.mecanum_stop()
            elif move_and_draw(frame, *target_box):
                print("Target reached.")
                return True

            ok, picture_jpg = cv2.imencode(".jpg", frame)
            if ok:
                image = Image(picture_jpg.tobytes())
                if handle is None:
                    handle = display(image, display_id=True)
                else:
                    handle.update(image)

            time.sleep(0.05)

        print("Time limit reached. Robot stopped.")
        return False

    finally:
        got.mecanum_stop()


### Run — one supervised approach


In [ ]:
reached_target = target_centralizing_approaching_Kp("bottle")
print("Reached target:", reached_target)


### Investigate

The robot may overshoot more than it did with colour tracking. Possible reasons
include slower detection, changing boxes, camera delay, and controller settings.
These are hypotheses to test.

### Modify

Change only one setting at a time. Try a smaller `Kp_steering` or a wider
`steering_dead_zone`, then record whether the movement improved.


---
# Session 9 — Capstone: the Lost Property Robot

**New idea:** run YOLO once per frame, then choose among several wanted names.

The robot searches for a lost item, approaches it, and announces the result.
The search and approach both have time limits.


### Detect any wanted item with one YOLO pass


In [ ]:
lost_items = ["bottle", "cup", "book"]

def find_any_target(frame, target_names, minimum_confidence=MIN_CONFIDENCE):
    results = model(frame, imgsz=MODEL_IMAGE_SIZE, verbose=False)

    best_name = None
    best_box = None
    best_confidence = -1.0

    for box in results[0].boxes:
        name = model.names[int(box.cls[0])]
        confidence = float(box.conf[0])

        is_wanted = name in target_names
        is_confident_enough = confidence >= minimum_confidence
        is_best_so_far = confidence > best_confidence

        if is_wanted and is_confident_enough and is_best_so_far:
            best_name = name
            best_box = box
            best_confidence = confidence

    if best_box is None:
        return None

    x1 = int(best_box.xyxy[0][0])
    y1 = int(best_box.xyxy[0][1])
    x2 = int(best_box.xyxy[0][2])
    y2 = int(best_box.xyxy[0][3])

    target_box = (x1, y1, x2 - x1, y2 - y1)
    return best_name, target_box, best_confidence


### Search safely


In [ ]:
def search_for_lost_item(max_search_seconds=20):
    import numpy as np

    def picture_bgr():
        picture_raw = got.read_camera_data()
        if picture_raw is None:
            return None
        return cv2.imdecode(
            np.frombuffer(picture_raw, np.uint8),
            cv2.IMREAD_COLOR,
        )

    started_at = time.monotonic()
    missing_frames = 0

    try:
        got.open_camera()
        got.play_audio_tts("I am looking for lost property.", 1, True)

        while time.monotonic() - started_at < max_search_seconds:
            frame = picture_bgr()

            if frame is None:
                missing_frames += 1
                got.mecanum_stop()
                if missing_frames >= 10:
                    print("Too many missing camera frames. Search stopped.")
                    return None
                time.sleep(0.1)
                continue

            missing_frames = 0
            detection = find_any_target(frame, lost_items)

            if detection is not None:
                item_name, target_box, confidence = detection
                got.mecanum_stop()
                print(f"I found a {item_name} with confidence {confidence:.2f}!")
                got.play_audio_tts(f"I found a {item_name}.", 1, True)
                return item_name

            got.mecanum_turn_speed(2, 10)
            time.sleep(0.1)

        print("Search time limit reached.")
        return None

    finally:
        got.mecanum_stop()


### Run the capstone


In [ ]:
found_item = search_for_lost_item()

if found_item is None:
    got.play_audio_tts("I could not find a lost item safely.", 1, True)
else:
    time.sleep(0.5)
    reached_item = target_centralizing_approaching_Kp(found_item)

    if reached_item:
        got.play_audio_tts(f"Here is your {found_item}. Job done!", 1, True)
    else:
        got.play_audio_tts("I stopped before reaching the item.", 1, True)


## Checks and reflection

- Can you point to the **sense**, **think**, and **act** parts of the capstone?
- What makes the robot stop?
- Why does `find_any_target` run the model only once per frame?
- Which number would you tune first if the robot turns too strongly?
- Why is box width only a clue about distance?

## Next step — only after the simple version is reliable

Deliver the item to an AprilTag zone using the previously tested
`check_point_navigation()` function from Term 1.
